In [15]:
import pandas as pd
import numpy as np
import faiss
from lightfm import LightFM
from lightfm.data import Dataset as LFM_Dataset
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix, eye
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm

DATA_PATH = 'data/'

# Load Data
def load_data(sample_size=50000, val_size=0.2):
    """Loads and merges all datasets, reducing dataset size and creating a validation set."""
    movies = pd.read_csv(f"{DATA_PATH}movies.csv")
    genome_scores = pd.read_csv(f"{DATA_PATH}genome_scores.csv")
    genome_tags = pd.read_csv(f"{DATA_PATH}genome_tags.csv")
    imdb_data = pd.read_csv(f"{DATA_PATH}imdb_data.csv")
    links = pd.read_csv(f"{DATA_PATH}links.csv")
    tags = pd.read_csv(f"{DATA_PATH}tags.csv")
    train = pd.read_csv(f"{DATA_PATH}train.csv").sample(sample_size, random_state=42)
    test = pd.read_csv(f"{DATA_PATH}test.csv")
    
    # Create validation set
    train_df, val_df = train_test_split(train, test_size=val_size, random_state=42)
    
    genome = genome_scores.merge(genome_tags, on='tagId')
    genome_pivot = genome.pivot(index='movieId', columns='tag', values='relevance').fillna(0)
    
    movies = movies.merge(imdb_data, on='movieId', how='left')
    movies = movies.merge(links, on='movieId', how='left')
    
    return train_df, val_df, test, movies, genome_pivot, tags

# Feature Engineering
def create_hybrid_features(movies, genome_df, tags):
    """Create hybrid content features."""
    movies['genres'] = movies['genres'].str.replace('|', ' ')
    genre_tfidf = TfidfVectorizer(max_features=500)
    genre_features = genre_tfidf.fit_transform(movies['genres'].fillna(''))
    
    # Process tags
    tags['tag'] = tags['tag'].astype(str)
    tag_data = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x.dropna())).reset_index()
    movies = movies.merge(tag_data, on='movieId', how='left')
    movies['tag'] = movies['tag'].fillna('')
    tag_tfidf = TfidfVectorizer(max_features=500)
    tag_features = tag_tfidf.fit_transform(movies['tag'])
    
    # Process genome scores
    genome_features = genome_df.reindex(movies['movieId']).fillna(0).values
    
    combined_features = hstack([genre_features, tag_features, genome_features])
    svd = TruncatedSVD(n_components=200)
    reduced_features = svd.fit_transform(combined_features)
    
    return reduced_features, movies

# Hybrid Model Class
class HybridRecommender:
    def __init__(self):
        self.cf_model = None
        self.content_index = None
        self.movie_map = None
        self.lgb_model = None
        self.scaler = None
        self.user_features = None
        self.item_features = None

    def fit(self, train_df, feature_matrix, movies, test_df):
        """Train collaborative filtering and content-based models."""
        # Determine the maximum userId and movieId across both training and test sets
        max_user_id = max(train_df['userId'].max(), test_df['userId'].max())
        max_movie_id = max(train_df['movieId'].max(), test_df['movieId'].max())
        
        self.cf_model = self.train_collaborative_model(train_df, max_user_id, max_movie_id)
        self.content_index = self.train_content_model(feature_matrix)
        
        # Create a mapping from movieId to index in feature_matrix
        self.movie_map = movies[['movieId']].reset_index(drop=True)
        self.movie_map['index'] = self.movie_map.index
        
        # Filter out movies in train_df that are not in feature_matrix
        train_df = train_df[train_df['movieId'].isin(self.movie_map['movieId'])]
        
        # Prepare training data
        movie_indices = train_df['movieId'].map(self.movie_map.set_index('movieId')['index']).values
        content_scores = self._compute_content_scores(feature_matrix, movie_indices)
        
        # Get collaborative filtering scores in bulk
        user_ids = train_df['userId'].values
        movie_ids = train_df['movieId'].values
        cf_scores = self.cf_model.predict(user_ids, movie_ids, user_features=self.user_features, item_features=self.item_features)
        
        # Combine features
        x_train = np.column_stack([content_scores, cf_scores])
        y_train = train_df['rating'].values
        
        # Normalize features
        self.scaler = StandardScaler()
        x_train = self.scaler.fit_transform(x_train)
        
        # Train LightGBM model
        self.lgb_model = lgb.LGBMRegressor()
        self.lgb_model.fit(x_train, y_train)
    
    def train_collaborative_model(self, train_df, max_user_id, max_movie_id):
        """Train LightFM collaborative filtering model."""
        dataset = LFM_Dataset()
        dataset.fit(train_df['userId'], train_df['movieId'])
        interactions, _ = dataset.build_interactions(train_df[['userId', 'movieId', 'rating']].values)
        
        # Construct sparse user and item feature matrices
        self.user_features = eye(max_user_id + 1, format='csr')
        self.item_features = eye(max_movie_id + 1, format='csr')
        
        model = LightFM(loss='warp')
        model.fit(interactions, user_features=self.user_features, item_features=self.item_features, epochs=10, num_threads=4)
        return model
    
    def train_content_model(self, feature_matrix):
        """Train FAISS index for fast similarity search."""
        feature_matrix = np.array(feature_matrix).astype('float32')
        self.scaler = StandardScaler()
        feature_matrix = self.scaler.fit_transform(feature_matrix)
        index = faiss.IndexFlatL2(feature_matrix.shape[1])
        index.add(feature_matrix)
        return index

    def _compute_content_scores(self, feature_matrix, movie_indices):
        """Compute content-based scores in bulk."""
        feature_vectors = feature_matrix[movie_indices]
        _, indices = self.content_index.search(feature_vectors, 10)
        return np.mean(indices, axis=1)

    def evaluate(self, val_df):
        """Evaluate model accuracy using RMSE and MAE on the validation set."""
        # Filter out movies in val_df that are not in feature_matrix
        val_df = val_df[val_df['movieId'].isin(self.movie_map['movieId'])]
        
        # Prepare validation data
        movie_indices = val_df['movieId'].map(self.movie_map.set_index('movieId')['index']).values
        content_scores = self._compute_content_scores(feature_matrix, movie_indices)
        
        # Get collaborative filtering scores in bulk
        user_ids = val_df['userId'].values
        movie_ids = val_df['movieId'].values
        cf_scores = self.cf_model.predict(user_ids, movie_ids, user_features=self.user_features, item_features=self.item_features)
        
        # Combine features
        x_val = np.column_stack([content_scores, cf_scores])
        x_val = self.scaler.transform(x_val)
        
        # Predict ratings
        y_pred = self.lgb_model.predict(x_val)
        y_true = val_df['rating'].values
        
        # Compute metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        print(f"Validation RMSE: {rmse}")
        print(f"Validation MAE: {mae}")
        return rmse, mae
    
    def predict_ratings(self, user_ids, movie_ids):
        """Predict ratings in bulk."""
        # Filter out movies in movie_ids that are not in feature_matrix
        valid_mask = np.isin(movie_ids, self.movie_map['movieId'])
        valid_user_ids = user_ids[valid_mask]
        valid_movie_ids = movie_ids[valid_mask]
        
        # Prepare content-based scores
        movie_indices = pd.Series(valid_movie_ids).map(self.movie_map.set_index('movieId')['index']).values
        content_scores = self._compute_content_scores(feature_matrix, movie_indices)
        
        # Get collaborative filtering scores in bulk
        cf_scores = self.cf_model.predict(valid_user_ids, valid_movie_ids, user_features=self.user_features, item_features=self.item_features)
        
        # Combine features
        x_pred = np.column_stack([content_scores, cf_scores])
        x_pred = self.scaler.transform(x_pred)
        
        # Predict ratings
        predicted_ratings = np.full(len(movie_ids), np.nan)  # Fill missing values with NaN
        predicted_ratings[valid_mask] = self.lgb_model.predict(x_pred)
        return predicted_ratings

# Generate Submission File with Vectorized Predictions
def generate_submission(recommender, test_df):
    """Generate predictions for the test set and save to a submission file using vectorized operations."""
    # Predict ratings in bulk
    test_df['predicted_rating'] = recommender.predict_ratings(test_df['userId'].values, test_df['movieId'].values)
    
    
        # Concatenate userId and movieId with an underscore
    test_df['id'] = test_df['userId'].astype(str) + '_' + test_df['movieId'].astype(str)
    
    # Save to CSV
    test_df[['id', 'predicted_rating']].to_csv('submission.csv', index=False)
    #test_df[['userId', 'movieId', 'predicted_rating']].to_csv('submission.csv', index=False)
    print(f"Submission file 'submission.csv' generated with {len(test_df)} rows.")

# Execution Pipeline
if __name__ == "__main__":
    train_df, val_df, test_df, movies_df, genome_df, tags_df = load_data(sample_size=30000, val_size=0.2)  # Reduce size
    feature_matrix, movies_df = create_hybrid_features(movies_df, genome_df, tags_df)
    recommender = HybridRecommender()
    recommender.fit(train_df, feature_matrix, movies_df, test_df)  # Pass test_df to fit
    joblib.dump(recommender, 'hybrid_recommender.pkl')
    recommender.evaluate(val_df)  # Evaluate model accuracy
    generate_submission(recommender, test_df)  # Generate submission file

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000354 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 2
[LightGBM] [Info] Start training from score 3.542687
Validation RMSE: 1.053548850944428
Validation MAE: 0.8338627790816769
Submission file 'submission.csv' generated with 5000019 rows.


In [16]:
sub = pd.read_csv('submission.csv')

In [17]:
sub

,id,predicted_rating
0,1_2011,3.824951
1,1_4144,3.774226
2,1_5767,3.431482
3,1_6711,3.435504
4,1_7318,3.763088
...,...,...
5000014,162541_4079,3.507452
5000015,162541_4467,3.409698
5000016,162541_4980,3.587593
5000017,162541_5689,3.719373
